# HealthBot: AI-Powered Patient Education System

MediTech Solutions — LangGraph Workflow Prototype

This notebook implements a LangGraph workflow that:
1. Asks the patient for a health topic
2. Searches Tavily for medical information
3. Summarizes results in patient-friendly language
4. Administers a comprehension quiz
5. Grades the answer with citations
6. Allows restart (with state reset) or exit

In [ ]:
# Load in the OpenAI key and Tavily key.
from dotenv import load_dotenv
import os

load_dotenv('config.env')

assert os.getenv('OPENAI_API_KEY') is not None
assert os.getenv('TAVILY_API_KEY') is not None

print("API keys loaded successfully.")

In [ ]:
from typing import TypedDict, Optional, Annotated
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

In [ ]:
class HealthBotState(TypedDict):
    """State object shared across all LangGraph nodes."""
    messages: Annotated[list, add_messages]
    topic: Optional[str]
    search_results: Optional[str]
    summary: Optional[str]
    quiz_question: Optional[str]
    patient_answer: Optional[str]
    grade_and_feedback: Optional[str]
    next_action: Optional[str]

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.3,
    api_key=os.getenv("OPENAI_API_KEY"),
)

tavily_tool = TavilySearchResults(
    max_results=5,
    search_depth="advanced",
    include_domains=[
        "mayoclinic.org",
        "webmd.com",
        "medlineplus.gov",
        "healthline.com",
        "nih.gov",
        "cdc.gov",
        "who.int",
        "medicalnewstoday.com",
        "clevelandclinic.org",
        "hopkinsmedicine.org"
    ],
    tavily_api_key=os.getenv("TAVILY_API_KEY")
)

print("OpenAI model and Tavily tool initialized.")

In [ ]:
def node_collect_topic(state: HealthBotState) -> HealthBotState:
    """Ask the patient what health topic they'd like to learn about."""
    print("\n" + "="*60)
    print("Welcome to HealthBot — Your Personal Health Educator")
    print("="*60)
    print("\nI'm here to help you understand medical topics in plain language.\n")

    topic = input("What health topic or medical condition would you like to learn about today? ")
    print(f"\nLooking up information on: '{topic}' ...")

    return {
        **state,
        "topic": topic,
        "messages": [HumanMessage(content=f"I want to learn about: {topic}")]
    }


def node_search_tavily(state: HealthBotState) -> HealthBotState:
    """Search Tavily for reputable medical information via LLM tool calling."""
    topic = state["topic"]
    llm_with_tools = llm.bind_tools([tavily_tool])

    search_prompt = [
        SystemMessage(content=(
            "You are a medical research assistant. Use the tavily_search_results_json tool "
            "to find accurate medical information from reputable health sources such as "
            "Mayo Clinic, WebMD, NIH, CDC, and WHO."
        )),
        HumanMessage(content=(
            f"Please search for detailed medical information about: {topic}. "
            "Find information about what it is, causes, symptoms, treatments, and prevention."
        ))
    ]

    response = llm_with_tools.invoke(search_prompt)
    search_results_text = ""

    if response.tool_calls:
        for tool_call in response.tool_calls:
            if tool_call["name"] == "tavily_search_results_json":
                raw_results = tavily_tool.invoke(tool_call["args"])
                for i, result in enumerate(raw_results, 1):
                    search_results_text += f"\n[Source {i}]: {result.get('url', 'Unknown')}\n"
                    search_results_text += f"{result.get('content', '')}\n"
                    search_results_text += "-" * 40 + "\n"

    print("Medical information retrieved from reputable sources.")

    return {
        **state,
        "search_results": search_results_text,
        "messages": state["messages"] + [
            response,
            ToolMessage(
                content=search_results_text,
                tool_call_id=response.tool_calls[0]["id"] if response.tool_calls else "search_001"
            )
        ]
    }


def node_summarize(state: HealthBotState) -> HealthBotState:
    """Summarize Tavily results into a 3-4 paragraph patient-friendly explanation."""
    topic = state["topic"]
    search_results = state["search_results"]

    summary_prompt = [
        SystemMessage(content=(
            "You are a compassionate patient educator. Rewrite the search results into a "
            "clear, patient-friendly summary. Rules:\n"
            "1. Use ONLY the information in the search results — no outside knowledge.\n"
            "2. Write in simple, plain English.\n"
            "3. Write exactly 3 to 4 paragraphs.\n"
            "4. Cover: what it is, causes/risk factors, symptoms/diagnosis, treatment/prevention.\n"
            "5. End with a reminder to consult a healthcare professional."
        )),
        HumanMessage(content=(
            f"Search results about '{topic}':\n\n{search_results}\n\n"
            "Summarize using ONLY the information above."
        ))
    ]

    response = llm.invoke(summary_prompt)
    return {**state, "summary": response.content, "messages": state["messages"] + [response]}


def node_present_summary(state: HealthBotState) -> HealthBotState:
    """Present summary and wait for patient to be ready for quiz."""
    topic = state["topic"]
    summary = state["summary"]

    print("\n" + "="*60)
    print(f"Health Information: {topic.title()}")
    print("="*60)
    print(f"\n{summary}\n")
    print("="*60)
    print("Note: For educational purposes only. Consult your healthcare provider.")
    print("="*60)

    input("\nPress ENTER when you've finished reading and are ready for a comprehension check...")
    print("\nGenerating a question based on what you just read...")
    return state


def node_generate_quiz(state: HealthBotState) -> HealthBotState:
    """Generate one quiz question using ONLY the summary."""
    summary = state["summary"]
    topic = state["topic"]

    quiz_prompt = [
        SystemMessage(content=(
            "Create ONE open-ended comprehension question for a patient. Rules:\n"
            "1. Base the question ONLY on the provided summary.\n"
            "2. The question must be answerable from the summary alone.\n"
            "3. Ask about an important concept.\n"
            "4. Output ONLY the question."
        )),
        HumanMessage(content=(
            f"Summary about '{topic}':\n\n{summary}\n\nWrite ONE quiz question."
        ))
    ]

    response = llm.invoke(quiz_prompt)
    return {**state, "quiz_question": response.content.strip(), "messages": state["messages"] + [response]}


def node_present_quiz(state: HealthBotState) -> HealthBotState:
    """Present quiz question and collect patient answer."""
    quiz_question = state["quiz_question"]

    print("\n" + "="*60)
    print("Comprehension Check")
    print("="*60)
    print(f"\n{quiz_question}\n")

    patient_answer = input("Your answer: ")
    print("\nEvaluating your response...")

    return {
        **state,
        "patient_answer": patient_answer,
        "messages": state["messages"] + [HumanMessage(content=patient_answer)]
    }


def node_grade_answer(state: HealthBotState) -> HealthBotState:
    """Grade the answer using ONLY the summary, with citations."""
    summary = state["summary"]
    quiz_question = state["quiz_question"]
    patient_answer = state["patient_answer"]

    grading_prompt = [
        SystemMessage(content=(
            "Grade the patient's quiz answer using ONLY the summary as your answer key.\n"
            "Assign a letter grade (A-F) and provide warm feedback with citations from the summary.\n"
            "Format:\n"
            "GRADE: [letter]\n"
            "FEEDBACK: [explanation with citations]\n"
            "CORRECT ANSWER: [brief correct answer from summary]"
        )),
        HumanMessage(content=(
            f"Summary:\n{summary}\n\n"
            f"Question: {quiz_question}\n\n"
            f"Patient Answer: {patient_answer}\n\n"
            "Grade this answer."
        ))
    ]

    response = llm.invoke(grading_prompt)
    return {**state, "grade_and_feedback": response.content, "messages": state["messages"] + [response]}


def node_present_grade(state: HealthBotState) -> HealthBotState:
    """Present grade and ask to continue or exit."""
    print("\n" + "="*60)
    print("Your Results")
    print("="*60)
    print(f"\n{state['grade_and_feedback']}\n")
    print("="*60)
    print("\nWhat would you like to do next?")
    print("  1. Learn about a new health topic")
    print("  2. Exit HealthBot")

    while True:
        choice = input("\nEnter 1 or 2: ").strip()
        if choice == "1":
            next_action = "new_topic"
            break
        elif choice == "2":
            next_action = "exit"
            break
        print("Please enter 1 or 2.")

    return {**state, "next_action": next_action}


def node_exit(state: HealthBotState) -> HealthBotState:
    """End the session."""
    print("\nThank you for using HealthBot! Goodbye!")
    return state


def node_reset_state(state: HealthBotState) -> HealthBotState:
    """Reset state for a new topic (privacy and accuracy)."""
    print("\nStarting a new session. Previous data cleared for your privacy.\n")
    return {
        "messages": [],
        "topic": None,
        "search_results": None,
        "summary": None,
        "quiz_question": None,
        "patient_answer": None,
        "grade_and_feedback": None,
        "next_action": None
    }

In [ ]:
def route_after_grade(state: HealthBotState) -> str:
    if state.get("next_action") == "new_topic":
        return "reset_state"
    return "exit"


def build_healthbot_graph():
    graph = StateGraph(HealthBotState)

    graph.add_node("collect_topic", node_collect_topic)
    graph.add_node("search_tavily", node_search_tavily)
    graph.add_node("summarize", node_summarize)
    graph.add_node("present_summary", node_present_summary)
    graph.add_node("generate_quiz", node_generate_quiz)
    graph.add_node("present_quiz", node_present_quiz)
    graph.add_node("grade_answer", node_grade_answer)
    graph.add_node("present_grade", node_present_grade)
    graph.add_node("reset_state", node_reset_state)
    graph.add_node("exit", node_exit)

    graph.set_entry_point("collect_topic")
    graph.add_edge("collect_topic", "search_tavily")
    graph.add_edge("search_tavily", "summarize")
    graph.add_edge("summarize", "present_summary")
    graph.add_edge("present_summary", "generate_quiz")
    graph.add_edge("generate_quiz", "present_quiz")
    graph.add_edge("present_quiz", "grade_answer")
    graph.add_edge("grade_answer", "present_grade")
    graph.add_conditional_edges(
        "present_grade",
        route_after_grade,
        {"reset_state": "reset_state", "exit": "exit"}
    )
    graph.add_edge("reset_state", "collect_topic")
    graph.add_edge("exit", END)

    return graph.compile()


healthbot = build_healthbot_graph()
print("HealthBot LangGraph workflow compiled successfully.")

In [ ]:
def run_healthbot():
    initial_state: HealthBotState = {
        "messages": [],
        "topic": None,
        "search_results": None,
        "summary": None,
        "quiz_question": None,
        "patient_answer": None,
        "grade_and_feedback": None,
        "next_action": None
    }

    print("\nStarting HealthBot Session...\n")
    return healthbot.invoke(initial_state)


run_healthbot()